In [36]:
import os
from typing import Annotated, List, TypedDict, Optional, Union
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

In [37]:
# The specialized framework for deep reasoning agents
from deepagents import create_deep_agent

In [38]:
load_dotenv()

True

In [39]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    paper_path: Optional[str]
    scientific_logic: Optional[str]
    implementation_code: Optional[str]

print("Environment and State initialized.")

Environment and State initialized.


In [40]:
llm= ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature= 0
)

In [41]:
research_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your job is to read research papers, extract the mathematical core, "
        "and explain the architecture in plain pseudocode. "
        "Use your planning tools to break down the paper analysis step-by-step."
    )
)
print("Deep Research Agent created.")

Deep Research Agent created.


In [42]:
import arxiv
import fitz  
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

In [43]:
# 1. arXiv Search and Download Tool
@tool
def download_research_paper(query: str)-> str:
    """
    Searches arXiv for a paper, downloads the PDF to the '../data/' folder, 
    and returns the local path to the file.
    """
    download_path= "../data/"
    os.makedirs(download_path, exist_ok= True)

    search= arxiv.Search(query= query, max_results= 1, sort_by= arxiv.SortCriterion.Relevance)
    paper= next(search.results())

    filename = f"{paper.title.replace(' ', '_').replace(':', '')}.pdf"
    full_path = paper.download_pdf(dirpath=download_path, filename=filename)
    
    return f"Paper downloaded to: {full_path}"

In [44]:
# 2. PDF Reading Tool
@tool
def read_pdf_content(file_path: str) -> str:
    """
    Opens a PDF file and extracts the text content. 
    Useful for reading downloaded research papers.
    """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [45]:
# 3. Web Search Tool (Tavily)
web_search = TavilySearchResults(k=3)

In [46]:
# Combine all tools
tools = [download_research_paper, read_pdf_content, web_search]
print("ArXiv, PDF, and Tavily tools defined.")

ArXiv, PDF, and Tavily tools defined.


In [47]:
# 1. Re-initialize the Researcher Agent with tools
# We give it our 'tools' list so it can search, download, and read.
researcher_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your mission is to find research papers and extract their implementation core. "
        "Step 1: Use 'download_research_paper' to get the PDF. "
        "Step 2: Use 'read_pdf_content' to understand the math and logic. "
        "Step 3: Output a structured explanation of the algorithm, variables, and pseudocode."
    )
)
print("Research Agent re-Linked with tools.")

Research Agent re-Linked with tools.


In [48]:
# trigger a "Deep" research session
inputs = {
    "messages": [
        ("user", "Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.")
    ]
}

config = {"configurable": {"thread_id": "research-task-1"}}
print("Starting Deep Research. Watch the agent plan and execute...")

for chunk in research_agent.stream(inputs, config):
    print(chunk)

Starting Deep Research. Watch the agent plan and execute...
{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={}, response_metadata={}, id='1fcc6fcc-289f-4532-84cc-e99579fdeda3')])}}
{'model': {'messages': [AIMessage(content='Okay, I will research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={'function_call': {'name': 'write_todos', 'arguments': '{"todos": [{"status": "in_progress", "content": "Research the QLoRA paper to understand the 4-bit NormalFloat quantization method."}, {"content": "Explain the 4-bit NormalFloat quantization method in detail.", "status": "pending"}]}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf70d-d38d-7d40-8441-7ea95286568c-0', tool_calls=[{'name': 'write_to

In [49]:
# This agent takes the research findings and plans the Code structure.
designer_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Architecture Designer. "
        "Your goal is to take research findings and create a high-level Design Spec. "
        "1. Define the Python classes and methods needed. "
        "2. Specify the exact mathematical formulas for each method. "
        "3. Decide on the libraries to use (e.g., torch, numpy). "
        "Output ONLY the design specification, not the actual code yet."
    )
)
print("Architecture Designer Agent created.")

Architecture Designer Agent created.


In [50]:
workflow = StateGraph(ResearchState)

# 1. Add Nodes
# IMPORTANT: We use .invoke()["messages"] to return a dict that matches our State
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Define the Flow (Initial)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", END)

# 3. Compile
app = workflow.compile()
print("Node logic fixed for Researcher & Designer.")

Node logic fixed for Researcher & Designer.


In [51]:
# The Coder Agent
# Its only job is to turn the Design Spec into high-quality, executable Python code.
coder_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt=(
        "You are the Lead Programmer for DeepForge. "
        "Your goal is to implement the Architecture Designer's specification. "
        "1. Write complete, well-documented Python code. "
        "2. Include a 'main' block or test case to verify the implementation. "
        "3. Ensure you follow research paper naming conventions for variables. "
        "Output ONLY the Python code inside a markdown block."
    )
)
print("Coder Agent created.")

Coder Agent created.


In [52]:
# 1. Add the Coder node to the existing workflow
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Re-route the Designer to the Coder instead of END
# We use a new edge to extend the chain
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", END)

# 3. Re-compile the Full Pipeline
deep_agent_pipeline = workflow.compile()
print("✅ Pipeline evolved! Coder added successfully.")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


✅ Pipeline evolved! Coder added successfully.


In [53]:
final_inputs= {
    "messages": [
        ("user", "Research the 'LoRA: Low-Rank Adaptation' paper and implement a simple LoRA linear layer in PyTorch.")
    ]
}
print("Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...")

# Run the pipeline
for event in deep_agent_pipeline.stream(final_inputs, config):
    # This will show you exactly which agent is working
    for node_name, output in event.items():
        print(f"\n--- [AGENT: {node_name.upper()}] ---")
        # We print the last message content from that agent
        print(output["messages"][-1].content)

Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_2336\2121793795.py:12: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper= next(search.results())



--- [AGENT: RESEARCHER] ---
The code for the LoRA linear layer and its test have been written. I am unable to execute the code. However, I can provide the implementation details based on the code I have written.

```
LoRA Linear Layer Implementation:

The LoRALinear class inherits from nn.Module.
It takes in_features, out_features, and rank as input.
It initializes two parameters, lora_A and lora_B, with shapes (rank, in_features) and (out_features, rank) respectively. lora_A is initialized with random values, and lora_B is initialized with zeros.
The forward function takes an input x and calculates the LoRA output as (x @ self.lora_A.transpose(0, 1) @ self.lora_B.transpose(0, 1)) * self.scaling.
The scaling factor is calculated as self.lora_rank**-0.5.

Variables:

in_features: The number of input features.
out_features: The number of output features.
rank: The rank of the LoRA adaptation.
lora_A: The A matrix of the LoRA adaptation.
lora_B: The B matrix of the LoRA adaptation.
scali